In [3]:
# Cell 1: Setup
"""
# ArcticDEM Elevation History Analysis
Extract and plot elevation time series at a point using ArcticDEM strip data.
"""

# Add the parent directory to path if needed
import sys
sys.path.append('..')

from config import OUTPUT_DIR, ARCHIVE_DIR
from analysis import run_elevation_history
from interactive_maps import select_point_interactive

import time
from ipyleaflet import (
    AwesomeIcon,
    FullScreenControl,
    Map,
    Marker,
    basemaps,
)
from IPython.display import clear_output, display
from ipywidgets import HTML, Button, HBox, Layout, VBox, widgets  # type: ignore


# STAC API needs to be defined in this file because it uses global coords (I think)
def stac_api(timeout=2):
    """
    Interactive map to select a single point with draggable marker.
    
    Displays a leaflet map where users can drag a marker and click
    'Confirm' to select coordinates.
    
    Parameters
    ----------
    timeout : int
        Maximum seconds to wait for user confirmation
        
    Returns
    -------
    tuple
        (longitude, latitude) in WGS84
    """
    global coords
    coords = (-54.2, 75)
    mid_lat = 75.0
    mid_lon = -54.2

    # Create a leaflet map widget to find an AOI for the spatial query of the STAC API
    m = Map(
        basemap=basemaps.Esri.WorldImagery,
        scroll_wheel_zoom=True,
        center=(mid_lat, mid_lon),
        zoom=6,
        layout=Layout(height="380px", width="700px"),
    )
    m.add_control(FullScreenControl())
    result_output = widgets.Output()

    # Create info display
    info_html = HTML(
        value="<b>Drag the marker, then click Confirm</b>",
        layout=Layout(padding="10px"),
    )

    # Create confirmation button
    confirm_button = Button(
        description="Confirm Location",
        button_style="success",
        disabled=False,
        tooltip="Click after placing marker",
    )

    # Create markers (initially slightly offset from center)
    centre_marker = Marker(
        # location=(center_y, center_x - 0.1),
        location=(mid_lat, mid_lon),  # Leaflet expects (lat, lon)
        draggable=True,
        name="Location",
        icon=AwesomeIcon(name="map-pin", marker_color="red"),
    )
    m.add_layer(centre_marker)

    # Update info display when marker moves
    def update_display(*args):
        # Remember: Leaflet uses (lat, lon) format, but we want to store as (lon, lat)
        centre_lat, centre_lon = centre_marker.location[0], centre_marker.location[1]
        info_html.value = (
            f"<b>Current Position:</b><br>"
            f"<span style='color:red'>Location:</span> Lon: {centre_lon:.3f}, Lat: {centre_lat:.3f}<br>"
        )

    # Handle confirmation
    def on_confirm(b):
        global coords
        # Store as (lon, lat) for consistency with GIS conventions
        centre_lat, centre_lon = centre_marker.location[0], centre_marker.location[1]
        with result_output:
            result_output.clear_output()
            print(f"Centre confirmed! (lon, lat): {centre_lon}, {centre_lat}")
            global coords
            coords = (centre_lon, centre_lat)

    # Connect callbacks
    confirm_button.on_click(on_confirm)
    centre_marker.observe(update_display, names=["location"])
    update_display()  # Initial display update

    # Display all components
    display(VBox([info_html, m, HBox([confirm_button]), result_output]))

    # Wait for confirmation with timeout
    start_time = time.time()
    while (time.time() - start_time) < timeout:
        time.sleep(0.1)

    clear_output(wait=True)  # Clean up the display
    return coords
print("✓ Modules loaded")

✓ Modules loaded


In [ ]:
# Cell 2: Configure Parameters
"""
## Configuration
Edit these parameters for your analysis.a
"""
# Analysis parameters
TIME_RANGE = "2009-01-01/2026-12-31"  # YYYY-MM-DD/YYYY-MM-DD
WINDOW_SIZE = 1                        # side size of the window in pixels (1 = single pixel, 3 = 3x3 window, etc.)
WINDOW_TYPE = 'square'                 # 'square' or 'cross'
COREG_MODE = 'none'                    # 'none' (faster, as it gets the elevation from compressed data), 'altim', or 'mosaic'

# Choose how to select coordinates
USE_INTERACTIVE_MAP = True             # True = click on map, False = type coordinates

print(f"Archive: {ARCHIVE_DIR}")
print(f"Time range: {TIME_RANGE}")
print(f"Window: {WINDOW_SIZE}x{WINDOW_SIZE} {WINDOW_TYPE}")
print(f"Coregistration: {COREG_MODE}")

Archive: /home/moralpom/luna/CPOM/archive/SATS/OPTICAL/ArcticDEM/strips/s2s041/2m/
Time range: 2009-01-01/2026-12-31
Window: 1x1 square
Coregistration: none
Output directory: /home/moralpom/luna/CPOM/moralpom/globe/data/ArcticDEM/elevation_histories/


In [ ]:
# Cell 3: Select Coordinates
"""
## Select Location
Choose a point for elevation history extraction.
"""
if USE_INTERACTIVE_MAP:
    print("Use the map to select your location...")
    coords = stac_api(
        timeout=2
    )
else:
    # Type coordinates manually
    coord_input = input("Enter coordinates as lon,lat (e.g. -54.2,75.0): ")
    coords = tuple(map(float, coord_input.split(',')))

Use the map to select your location...


In [7]:
# Cell 4: Run Analysis
"""
## Process Elevation History
Query STAC API and extract elevations from all available DEMs.
"""

history = run_elevation_history(
    archdir=ARCHIVE_DIR,
    coords=coords,
    time_range=TIME_RANGE,
    window_size=WINDOW_SIZE,
    window_type=WINDOW_TYPE,
    coreg_mode=COREG_MODE
)

print("\n✓ Analysis complete!")
print(f"Processed {len(history['dates'])} DEMs")
print(f"Valid elevations: {sum(1 for m in history['metadata'] if m.get('valid', False))}")

Coordinates: (-47.900, 70.289)
Time range: 2009-01-01/2026-12-31
Found 16 StripDEMs
After cloud filter (<20%): 15 DEMs
After xtrack filter: 15 DEMs

=== Processing elevation history for 15 DEMs ===
Window: 1x1 square
Coordinates (EPSG:3413): -109077.2, -2152925.5

Processing 1/15: WV02_20240717_10300101011BD100_103001010181F200
  Elevation: 1884.0 ± 0.0 m (n=1)

Processing 2/15: WV02_20240321_10300100F726F600_10300100F8872800
  Elevation: 1881.5 ± 0.0 m (n=1)

Processing 3/15: WV01_20230823_10200100DDCAD500_10200100DE4E6000
  Elevation: 1882.4 ± 0.0 m (n=1)

Processing 4/15: WV02_20210309_10300100BB65B300_10300100BB97AE00
  Elevation: 1883.4 ± 0.0 m (n=1)

Processing 5/15: WV02_20200617_10300100A566A700_10300100A8303000
  Elevation: 1879.8 ± 0.0 m (n=1)

Processing 6/15: WV03_20190729_104001004E3D6800_104001004F09EB00
  Elevation: 1879.2 ± 0.0 m (n=1)

Processing 7/15: WV03_20190724_104001004F75E200_104001004E755100
  Elevation: 1881.0 ± 0.0 m (n=1)

Processing 8/15: WV02_20170723_1030